# Notebook 1: Spark Fundamentals for Genomics

This notebook teaches core Spark concepts you'll need for the main exercise.

**Learning Objectives:**
1. Understand lazy evaluation and query plans
2. Work with partitions and data distribution
3. Use core DataFrame operations (filter, select, groupBy, join)
4. Identify operations that cause shuffles
5. Apply caching strategies effectively

**Estimated time: ~90 minutes**

**Prerequisites**: Completed Notebook 00

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
import pandas as pd
import time

spark = SparkSession.builder \
    .appName("BINFX410_Spark_Basics") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

data_dir = "/workspace/data"
if not os.path.exists(data_dir):
    data_dir = os.path.join(os.path.dirname(os.getcwd()), "data")

variants = spark.read.csv(
    os.path.join(data_dir, "variants_small.csv"),
    header=True, inferSchema=True
)
print(f"Loaded {variants.count()} variants")
print(f"Spark UI: http://localhost:4040")

---
## Section 1: Lazy Evaluation (~15 min)

Spark uses **lazy evaluation**: when you call operations like `filter()` or `select()`,
Spark doesn't actually process any data. Instead, it builds an **execution plan**.
Data is only processed when you call an **action** like `count()`, `show()`, or `collect()`.

**Transformations** (lazy - build a plan):
- `filter()`, `select()`, `groupBy()`, `join()`, `withColumn()`

**Actions** (eager - trigger computation):
- `count()`, `show()`, `collect()`, `write()`

In [ ]:
# These are transformations - nothing executes yet!
step1 = variants.filter(F.col("QUAL") >= 30)
step2 = step1.select("CHROM", "POS", "REF", "ALT", "QUAL")
step3 = step2.filter(F.col("CHROM") == "chr1")

print("No computation happened yet! Spark just built a plan.")
print(f"Type of step3: {type(step3)}")

In [ ]:
# See what Spark PLANS to do (without executing)
print("Execution plan:")
step3.explain()

The plan reads bottom-to-top. Notice how Spark combines the two filters into a single
operation - this is the **Catalyst optimizer** at work. Lazy evaluation allows Spark to
optimize the entire pipeline before running it.

In [ ]:
# NOW computation happens when we call an action
start = time.time()
result = step3.count()
duration = time.time() - start

print(f"Found {result} variants on chr1 with QUAL >= 30")
print(f"Execution time: {duration:.3f} seconds")

In [ ]:
# Extended explain shows the optimization steps
print("Extended plan (shows Parsed -> Analyzed -> Optimized -> Physical):")
step3.explain(True)

---
## Section 2: Partitions (~20 min)

Spark splits data into **partitions** - chunks that can be processed independently
on different CPU cores (or different machines in a cluster). The number of partitions
directly affects parallelism.

In [ ]:
print(f"Number of partitions: {variants.rdd.getNumPartitions()}")

In [ ]:
# Repartition: change the number of partitions
# repartition() does a FULL shuffle (expensive but balances data)
repartitioned = variants.repartition(4)
print(f"After repartition(4): {repartitioned.rdd.getNumPartitions()} partitions")

# Repartition by column - hash partitioning
by_chrom = variants.repartition("CHROM")
print(f"After repartition('CHROM'): {by_chrom.rdd.getNumPartitions()} partitions")

# Coalesce: reduce partitions WITHOUT a shuffle (cheaper)
coalesced = variants.coalesce(2)
print(f"After coalesce(2): {coalesced.rdd.getNumPartitions()} partitions")

**Key difference:**
- `repartition(n)`: Full shuffle, can increase or decrease partitions, balances data evenly
- `coalesce(n)`: No shuffle, can only *decrease* partitions, may create uneven sizes

In [ ]:
# See how data is distributed across partitions
def show_partition_distribution(df, name="DataFrame"):
    partition_counts = df.withColumn("partition_id", F.spark_partition_id()) \
        .groupBy("partition_id").count().orderBy("partition_id")
    print(f"\n{name} - Rows per partition:")
    partition_counts.show()

show_partition_distribution(variants, "Original")
show_partition_distribution(by_chrom, "Repartitioned by CHROM")

---
## Section 3: Common Operations (~25 min)

### Filtering

In [ ]:
# filter() and where() are identical
high_qual = variants.filter(F.col("QUAL") >= 30)
high_qual_v2 = variants.where(F.col("QUAL") >= 30)

# Multiple conditions (use & for AND, | for OR, ~ for NOT)
strict_filter = variants.filter(
    (F.col("QUAL") >= 30) &
    (F.col("DP") >= 10) &
    (F.col("GQ") >= 20)
)
print(f"Original:            {variants.count()}")
print(f"After strict filter: {strict_filter.count()}")

### Selecting and Creating Columns

In [ ]:
# Select columns
subset = variants.select("CHROM", "POS", "QUAL")
subset.show(5)

# Create new columns with withColumn + when/otherwise
variants_with_class = variants.withColumn(
    "quality_class",
    F.when(F.col("QUAL") >= 50, "HIGH")
     .when(F.col("QUAL") >= 30, "MEDIUM")
     .otherwise("LOW")
)
variants_with_class.select("CHROM", "POS", "QUAL", "quality_class").show(10)

### Aggregating

In [ ]:
# GroupBy with multiple aggregations
chrom_stats = variants.groupBy("CHROM").agg(
    F.count("*").alias("variant_count"),
    F.round(F.mean("QUAL"), 2).alias("mean_qual"),
    F.round(F.mean("DP"), 2).alias("mean_depth"),
    F.sum(F.when(F.col("FILTER") == "PASS", 1).otherwise(0)).alias("pass_count")
)
chrom_stats.orderBy("variant_count", ascending=False).show()

### Sorting

In [ ]:
# Order by multiple columns
variants.orderBy(F.col("CHROM"), F.col("POS")).show(10)

# Descending order
variants.orderBy(F.col("QUAL").desc()).show(5)

### Joining

In [ ]:
# Load gene annotations
genes = spark.read.csv(
    os.path.join(data_dir, "genes.bed"),
    sep="\t", header=False,
    schema="chrom STRING, start INT, end INT, gene_name STRING, strand STRING"
)
print(f"Loaded {genes.count()} gene annotations")
genes.show(5)

# Positional join: variant is in a gene if CHROM matches and start <= POS <= end
variant_genes = variants.join(
    genes,
    (variants.CHROM == genes.chrom) &
    (variants.POS >= genes.start) &
    (variants.POS <= genes.end),
    "left"
)
in_gene = variant_genes.filter(F.col("gene_name").isNotNull()).count()
print(f"\nVariants in genes: {in_gene}")
print(f"Intergenic variants: {variants.count() - in_gene}")

---
## Section 4: Understanding Shuffles (~20 min)

A **shuffle** is when Spark redistributes data across partitions. This happens for operations
like `groupBy`, `join`, and `repartition`. Shuffles are expensive because they involve
writing data to disk and transferring it between executors.

| Narrow (no shuffle) | Wide (shuffle) |
|---|---|
| filter(), where() | groupBy() |
| select(), withColumn() | join() |
| map operations | repartition() |
| coalesce() | orderBy() / sort() |

In [ ]:
# Narrow transformation - no shuffle (no "Exchange" in plan)
filtered = variants.filter(F.col("QUAL") >= 30)
print("Plan for filter (narrow transformation):")
filtered.explain()

start = time.time()
filtered.count()
print(f"Time: {time.time() - start:.3f}s")

In [ ]:
# Wide transformation - shuffle required (look for "Exchange" in plan)
grouped = variants.groupBy("CHROM").count()
print("Plan for groupBy (wide transformation):")
grouped.explain()

start = time.time()
grouped.collect()
print(f"Time: {time.time() - start:.3f}s")

In [ ]:
# Minimizing shuffle cost: filter BEFORE groupBy

# Approach 1: groupBy all data, then filter results
start = time.time()
approach1 = variants.groupBy("CHROM").count().filter(F.col("count") > 10)
approach1.collect()
time1 = time.time() - start

# Approach 2: filter data first, then groupBy (less data shuffled)
start = time.time()
approach2 = variants.filter(F.col("QUAL") >= 30).groupBy("CHROM").count()
approach2.collect()
time2 = time.time() - start

print(f"GroupBy then filter: {time1:.3f}s")
print(f"Filter then groupBy: {time2:.3f}s")
print("\nFiltering before groupBy reduces the amount of data shuffled!")

---
## Section 5: Caching (~10 min)

When you reuse a DataFrame multiple times, Spark **recomputes** it from scratch each time
(because of lazy evaluation). **Caching** stores the result in memory so subsequent
uses are instant.

**When to cache:**
- DataFrame is used multiple times in different operations
- After an expensive computation (join, complex filter)

**When NOT to cache:**
- DataFrame is only used once
- DataFrame is very large (might not fit in memory)
- You're running low on memory

In [ ]:
# Without caching - Spark recomputes each time
filtered = variants.filter(F.col("QUAL") >= 30)

start = time.time()
filtered.count()
time_uncached_1 = time.time() - start

start = time.time()
filtered.groupBy("CHROM").count().collect()
time_uncached_2 = time.time() - start

print(f"Uncached - count: {time_uncached_1:.3f}s, groupBy: {time_uncached_2:.3f}s")

# With caching - computed once, stored in memory
filtered.cache()
filtered.count()  # triggers caching

start = time.time()
filtered.count()
time_cached_1 = time.time() - start

start = time.time()
filtered.groupBy("CHROM").count().collect()
time_cached_2 = time.time() - start

print(f"Cached   - count: {time_cached_1:.3f}s, groupBy: {time_cached_2:.3f}s")

# Always clean up cached data when done
filtered.unpersist()

---
## Summary

Key concepts covered:

1. **Lazy Evaluation**: Transformations build a plan; actions trigger execution
2. **Partitions**: Data is split into chunks for parallel processing
3. **Core Operations**: filter, select, groupBy, join - similar to pandas but distributed
4. **Shuffles**: Expensive data redistribution; minimize by filtering before aggregating
5. **Caching**: Store reused DataFrames in memory to avoid recomputation

**Next**: Open `02_variant_qc_exercise.ipynb` to apply these concepts!

In [ ]:
spark.stop()
print("SparkSession stopped. Ready for the next notebook!")